<img src="starting_point.jpg" width="800">

Notebook for using the Surrounding self-energies for later use in the project.
Based on [Alan's notebook](../../resources/SE_Graphene_Example-TB.ipynb)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import sisl
from sisl import Hamiltonian
import ase
from ase.visualize import view
from tqdm.auto import tqdm

# Create basis for pristine graphene layer (primitive unit cell)

In [ ]:
bond = 1.43
gr = sisl.geom.graphene(bond, "C")
gr.plot(axes="xy")

# Make hamiltonian

In [ ]:
Ham0 = Hamiltonian(gr)
print(Ham0)

# Define TB

In [ ]:
bond = 1.43
r = (0.1*bond, bond + 0.01)
t = (0, -2.7)
Ham0.construct([r, t])
print(Ham0.Hk(format="array"))

In [ ]:
Ham = Ham0.tile(1,0).tile(1,1)

In [ ]:
bs_points = [[0., 0., 0.], [1 / 3, 2 / 3, 0], [0, 0.5, 0],[0., 0., 0.]]
bs_div = 300
bs_labels = [r'$\Gamma$', r"$K$", r"$M$",r'$\Gamma$']

band = sisl.BandStructure(Ham, points=bs_points, divisions=bs_div, names=bs_labels)
lineark, kticks, klabels = band.lineark(True)

fig, ax = plt.subplots()

bs = np.empty(shape=(len(band),2))
for ik, eigh in enumerate(band.apply.eigh()):
    bs[ik] = eigh

ax.plot(lineark, bs, label="TB")
for tick in kticks:
    ax.axvline(x=tick, lw=0.1, color="k")
ax.set(xlim=(kticks[0], kticks[-1]), ylim=(-15, 15))
ax.axhline(y=0, color="k", lw=0.1)
ax.set_xticks(kticks)
ax.set_xticklabels(klabels)

None

# sisl calculations

In [ ]:
print(Ham0)

In [ ]:
eta = 0.001* 1j
dE=0.1
Emax = 3.0
Emin = -Emax
energies = np.arange(Emin, Emax + dE, dE)

# Real Space Self Energies
N0 = 12
N1 = 12
nk1 = int(np.ceil(3*900/N1))

rse = sisl.RealSpaceSE(Ham, 0, 1, (N0, N1, 1))
rse.setup(eta= 0.001, bz=sisl.MonkhorstPack(Ham0, [1, nk1, 1]))
HamNN = Ham0.tile(N0, 0).tile(N1, 1)
geomNN = HamNN.geometry
elec_indices = rse.real_space_coupling(geomNN)[1]
nC = len(elec_indices)   # number of coupling atoms
HamNN.set_nsc([1,1,1]) # remove al pbc

# indexes sort
all_atoms=np.arange(0,HamNN.na)
inside_atoms = np.delete(all_atoms, elec_indices, axis=None)
alist = np.concatenate([elec_indices, inside_atoms])

# Reordering Hamiltonain
HamNN_reorder=HamNN.sub(alist)
HamNN_reorder.reduce()
HamNN_reorder.set_nsc([1,1,1]) # remove al pbc
H_sub = HamNN_reorder.Hk(format="array")   
S_sub = HamNN_reorder.Sk(format="array")

In [ ]:
def calc_GF(E,eta,rse,H_sub,S_sub,alist=None):
    z = E +  eta
    A = z*S_sub - H_sub 
    RSE = rse.self_energy(z)
    if alist is not None:
        RSE = RSE[np.ix_(alist, alist)]
        A[0:len(alist),0:len(alist)] -= RSE  
    else: 
        A -= RSE
    G = np.linalg.inv(A)   
    ldos = - (1.0 / np.pi) * np.imag(np.diag(G@S_sub))
    # dos = - (1.0 / np.pi) * np.imag(np.trace(G@S_sub)) 
    return ldos


def loop_GF(energies,eta,rse,H_sub,S_sub,alist=None):
    # Observables
    N = H_sub.shape[0]
    # DOS = np.zeros_like(energies, dtype=float)
    LDOS = np.zeros((N, len(energies)), dtype=float)  

    # Loop
    for iE, E in enumerate(tqdm(energies, desc='DOS')):
        ldos = calc_GF(E,eta,rse,H_sub,S_sub,alist=alist)
        # DOS[iE] = np.sum(ldos)
        LDOS[:,iE] = ldos
    return LDOS

In [ ]:
import time
t0 = time.perf_counter()
LDOS = loop_GF(energies,eta,rse,H_sub,S_sub,alist=alist)
t1 = time.perf_counter()
print(f"Total runtime: {t1 - t0:.2f} seconds")
print(f"Time per energy: {(t1 - t0)/len(energies):.4f} s")

LDOS_dev_OG = LDOS[nC:,:]
DOS_dev_OG = LDOS_dev_OG.sum(axis=0)



In [ ]:
np.savez("alans_calcs.npz", ldos=LDOS_dev_OG, dos=DOS_dev_OG, E=energies)

# Tile the primitive unit cell hamiltonian

In [ ]:
# geomNN.plot(axes="xy")

# Remove from the device if part of the electrode

In [ ]:
# Ham_elec = HamNN.sub(elec_indices)
# Ham_inside = HamNN.sub(inside_atoms)
# Ham_inside.geometry.plot(axes="xy")

In [ ]:
# from sisl.viz import merge_plots
# plots = [
#     Ham_inside.geometry.plot(axes="xy", atoms_style={"color":"gray"}), # the device atoms
#     Ham_elec.geometry.plot(axes="xy", atoms_style=({"color":"red"}))
# ]
# merge_plots( *plots, composite_method="multiple"
# )

# View atoms to show atom index labels before reordering

In [ ]:
# view(HamNN.geometry.to.ase())
# view(Ham_inside.geometry.to.ase())

# View the reordered atoms

In [ ]:
# view(HamNN_reorder.geometry.to.ase())